# Phase 1: Radiation & Plasma — Research Charts

Generated charts for the Gates of Truth Phase 1 simulation architecture report.
See `phase1-radiation-plasma-truth.md` for the full report.

> Run all cells (`Kernel → Run All`) to render charts. Requires `plotly` and `numpy`.

In [1]:
import numpy as np
import plotly.graph_objects as go

## 1. Stellar SED Comparison (O5 / G2 / M5)

Blackbody-approximated spectral energy distributions. O5 (40,000 K) peaks in far-UV; G2 (5,800 K) in visible; M5 (3,000 K) in NIR. Drives `c/sed-bands` in `stellar-sed-system`.

In [2]:
def planck(lam_m, T):
    h, c, kB = 6.62607015e-34, 2.99792458e8, 1.380649e-23
    return (2*h*c**2/lam_m**5) / (np.exp(h*c/(lam_m*kB*T)) - 1)

lam_nm = np.logspace(2, 4, 300)
lam_m  = lam_nm * 1e-9

fig = go.Figure()
for label, T in [("O5  (40 kK)", 40000), ("G2  (5.8 kK)", 5800), ("M5  (3 kK)", 3000)]:
    spec = planck(lam_m, T)
    fig.add_trace(go.Scatter(x=lam_nm, y=np.log10(spec/spec.max()), mode='lines', name=label, line=dict(width=3)))

fig.update_layout(title="Stellar SED Comparison — O5 / G2 / M5",
                  xaxis_title="Wavelength (nm)", yaxis_title="log10(normalised flux)",
                  legend=dict(orientation='h', y=1.1, x=0.5, xanchor='center'))
fig.update_xaxes(type="log")
fig.show()

## 2. Condensation Sequence

Condensation temperatures for key nebular materials. Controls solid surface density jumps and ice-line positions in `disk-evolution-system`.

In [3]:
materials = ["Al-Ca oxides", "Fe-Ni metal", "Silicates", "Water ice", "CO2 ice", "CH4/NH3 ices"]
temps     = [1700, 1350, 1200, 170, 90, 60]
colors    = ["#e45756","#f58518","#eeca3b","#72b7b2","#54a24b","#4c78a8"]

fig = go.Figure(go.Bar(x=materials, y=temps, marker_color=colors, text=temps, textposition='outside'))
fig.update_layout(title="Condensation Sequence — Nebular Disk",
                  xaxis_title="Material", yaxis_title="Condensation T (K)")
fig.update_xaxes(tickangle=-20)
fig.show()

## 3. Streaming Instability Onset (St, Z)

Critical solid-to-gas ratio Z as a function of Stokes number St. Above threshold, streaming instability drives rapid planetesimal formation. Drives `planetesimal-formation-system` onset condition.

In [4]:
St    = np.logspace(-3, 0, 300)
Zlam  = 0.02 * (St / 0.1)**0.30
Zturb = 0.04 * (St / 0.1)**0.25

fig = go.Figure()
fig.add_trace(go.Scatter(x=St, y=Zlam,  mode='lines', name="Laminar threshold",  line=dict(width=3)))
fig.add_trace(go.Scatter(x=St, y=Zturb, mode='lines', name="Turbulent threshold", line=dict(width=3, dash='dash')))
fig.update_layout(title="Streaming Instability Onset in (St, Z) Space",
                  xaxis_title="Stokes number (St)", yaxis_title="Critical Z (solid/gas ratio)",
                  legend=dict(orientation='h', y=1.1, x=0.5, xanchor='center'))
fig.update_xaxes(type="log")
fig.show()

## 4. Impact Regime Diagram

Schematic partition of cratering, disruption, and merging regimes in impactor mass vs velocity space. Drives regime classification in `collision-system`.

In [5]:
fig = go.Figure()
for regime, vels, masses, sym in [
    ("Cratering",  [1e3,3e3,1e4], [1e18,1e20,1e22], "circle"),
    ("Disruption", [5e3,1e4,3e4], [1e20,1e22,1e24], "square"),
    ("Merging",    [2e4,5e4,1e5], [1e22,1e24,1e26], "diamond"),
]:
    fig.add_trace(go.Scatter(x=vels, y=masses, mode='markers', marker=dict(size=18, symbol=sym), name=regime))

fig.update_layout(title="Impact Regime Diagram",
                  xaxis_title="Impact velocity (m/s)", yaxis_title="Impactor mass (kg)",
                  legend=dict(orientation='h', y=1.1, x=0.5, xanchor='center'))
fig.update_xaxes(type="log")
fig.update_yaxes(type="log")
fig.show()

## 5. Planet Formation Timeline

Growth track from interstellar dust to a finished planet. Annotates phase transitions for `phase0` promotion logic.

In [6]:
stages = ["Dust", "Pebbles", "Planetesimals", "Protoplanets", "Planet"]
log_t  = [3, 4, 5, 6, 7]
masses = [1e-6, 1e-3, 1e-2, 0.1, 1.0]

fig = go.Figure(go.Scatter(x=log_t, y=masses, mode='lines+markers+text',
    text=stages, textposition='top center', line=dict(width=3), marker=dict(size=12)))
fig.update_layout(title="Planet Formation Timeline",
                  xaxis_title="log\u2081\u2080 time (yr)", yaxis_title="Mass (M\u2295)")
fig.update_yaxes(type="log")
fig.show()

## 6. Tectonic Regime vs Planet Mass & Surface Temperature

Stagnant lid, episodic, and mobile plate regimes. Most planets are stagnant lid. Drives `tectonic-system` regime selection.

In [7]:
fig = go.Figure()
for regime, ms, ts in [
    ("Stagnant lid",  [0.1,0.3,0.5,1.0], [200,220,240,500]),
    ("Episodic",      [0.5,1.0,2.0],     [300,350,420]),
    ("Mobile plates", [1.0,2.0,3.0],     [280,320,360]),
]:
    fig.add_trace(go.Scatter(x=ms, y=ts, mode='markers', marker=dict(size=16), name=regime))

fig.update_layout(title="Tectonic Regime vs Planet Mass & Surface Temperature",
                  xaxis_title="Mass (M\u2295)", yaxis_title="Surface temperature (K)",
                  legend=dict(orientation='h', y=1.1, x=0.5, xanchor='center'))
fig.show()